In [ ]:
"""
Análisis Tecno-Económico del Agregador en España
Impacto en la Red Local: Gestión Individual (UEM) vs Gestión Coordinada (CEM)

Este script genera mediante modelado matemático sintético (funciones gaussianas y senoidales)
tres perfiles de demanda para ilustrar el problema del efecto rebote o pico de coincidencia
en redes de distribución sin coordinación, frente a la mitigación lograda por un agregador.

Autor: Alberto Zafra Muñoz
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
from typing import Tuple

# ==========================================
# 1. MODELADO MATEMÁTICO DE ESCENARIOS
# ==========================================
def generar_perfiles_demanda(resolucion: int = 500) -> Tuple[list, np.ndarray, np.ndarray, np.ndarray]:
    """
    Genera las series temporales de demanda eléctrica simulando tres escenarios operativos.
    
    Parámetros:
        resolucion (int): Número de puntos para la discretización temporal (suavizado de curva).
        
    Retorna:
        Tuple: (eje temporal, demanda base, demanda UEM, demanda CEM)
    """
    # Eje temporal continuo de 24 horas
    horas_num = np.linspace(0, 24, resolucion)
    fecha_base = datetime(2024, 1, 1)
    horas_dt = [fecha_base + timedelta(hours=h) for h in horas_num]

    # A. Demanda Base Inelástica (Comportamiento pasivo)
    # Modelada mediante una componente senoidal diurna y un pico gaussiano vespertino
    componente_diurna = 0.5 * np.sin(np.pi * (horas_num - 8) / 12)
    pico_vespertino = 1.5 * np.exp(-0.1 * (horas_num - 20)**2)
    demanda_base = 2.0 + componente_diurna + pico_vespertino

    # B. Escenario UEM (Uncoordinated Energy Management)
    # Simula la activación simultánea de cargas flexibles (VE, Baterías) al inicio del periodo tarifario valle (02:00h)
    amplitud_uem = 5.0
    hora_central_uem = 2.5
    dispersion_uem = 3.0
    pico_coincidencia = amplitud_uem * np.exp(-dispersion_uem * (horas_num - hora_central_uem)**2)
    demanda_uem = demanda_base + pico_coincidencia

    # C. Escenario CEM (Coordinated Energy Management - Planta de Energía Virtual)
    # Simula el reparto óptimo (MILP) de la misma cantidad de energía a lo largo de la ventana temporal
    amplitud_cem = 1.8
    hora_central_cem = 4.5
    dispersion_cem = 0.2
    carga_repartida = amplitud_cem * np.exp(-dispersion_cem * (horas_num - hora_central_cem)**2)
    demanda_cem = demanda_base + carga_repartida

    return horas_dt, demanda_base, demanda_uem, demanda_cem

# ==========================================
# 2. REPRESENTACIÓN GRÁFICA ACADÉMICA
# ==========================================
def visualizar_comparativa(horas_dt: list, dem_base: np.ndarray, dem_uem: np.ndarray, dem_cem: np.ndarray):
    """
    Genera y formatea la figura comparativa de los escenarios de gestión de flexibilidad.
    """
    plt.figure(figsize=(12, 6))
    plt.style.use('seaborn-v0_8-whitegrid')

    # 1. Trazado de series de potencia
    plt.plot(horas_dt, dem_base, color='gray', linestyle='--', linewidth=2, label='Demanda Base (Inelástica)')
    
    plt.plot(horas_dt, dem_uem, color='#d62728', linewidth=3, label='UEM (No Coordinada) - Pico de Coincidencia')
    plt.fill_between(horas_dt, dem_uem, dem_base, where=(dem_uem > dem_base), color='#d62728', alpha=0.15)

    plt.plot(horas_dt, dem_cem, color='#2ca02c', linewidth=3, label='CEM (Agregador) - Flexibilidad Optimizada')
    plt.fill_between(horas_dt, dem_cem, dem_base, where=(dem_cem > dem_base), color='#2ca02c', alpha=0.25)

    # 2. Anotaciones técnicas (Callouts)
    plt.annotate(
        'Sobrecarga del\nTransformador Local',
        xy=(datetime(2024, 1, 1, 2, 30), 7.0),
        xytext=(datetime(2024, 1, 1, 4, 0), 6.5),
        arrowprops=dict(facecolor='#d62728', shrink=0.05, width=1.5, headwidth=8),
        fontsize=11, color='#d62728', fontweight='bold', ha='left'
    )

    plt.annotate(
        'Peak Shaving\n(Desplazamiento de Carga)',
        xy=(datetime(2024, 1, 1, 4, 30), 3.8),
        xytext=(datetime(2024, 1, 1, 8, 0), 4.5),
        arrowprops=dict(facecolor='#2ca02c', shrink=0.05, width=1.5, headwidth=8),
        fontsize=11, color='#2ca02c', fontweight='bold', ha='left'
    )

    # 3. Formateo de ejes y leyendas
    ax = plt.gca()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
    plt.gcf().autofmt_xdate(rotation=0, ha='center')

    plt.title('Impacto de la Gestión de Flexibilidad en la Red: UEM vs CEM', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Hora del día', fontsize=12)
    plt.ylabel('Potencia Demandada (kW)', fontsize=12)
    
    plt.ylim(0, dem_uem.max() + 1)
    plt.xlim(horas_dt[0], horas_dt[-1])

    plt.legend(loc='upper right', frameon=True, shadow=True, fontsize=11)
    plt.tight_layout()
    plt.show()

# ==========================================
# 3. BLOQUE PRINCIPAL DE EJECUCIÓN
# ==========================================
if __name__ == "__main__":
    print("Generando datos sintéticos de simulación...")
    t, base, uem, cem = generar_perfiles_demanda()
    
    print("Generando visualización gráfica...")
    visualizar_comparativa(t, base, uem, cem)